# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labanaprince72-a11y/internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This notebook turns the transparent Week-4 action score into a practical, human-reviewed content playbook. The queue is a prioritization aid for an anonymized snapshot, not an autonomous publishing system.

The Week-5 model and Week-6 audit provide guardrails: on the grouped client holdout, logistic regression measured Precision@50 of 0.620 versus 0.440 for the Week-4 baseline, but the result is directional decision-support rather than causal evidence. The queue therefore keeps reason codes, limits, and human gates visible.


## 1. Ranked actions + reason codes

The queue starts with the existing transparent score from `baseline_action_score.csv`. Its reason codes are intentionally legible: a reviewer can see whether a row is a visible low-CTR page, stale visible content, a low-visibility monitor candidate, or a page that needs search-fit research.

Priority tiers are operational suggestions, not labels of content quality:

- **P1 — review this week:** score 5–6; two or more strong review signals.
- **P2 — review next cycle:** score 3–4; one clear signal or a moderate combination.
- **P3 — monitor/sample:** score 1–2; weak directional evidence.
- **P4 — hold:** score 0; do not spend editorial effort without new evidence.

The action mapping preserves the source reason code and adds a next step, effort estimate, and value proxy. `clicks × CPC` is a captured click-equivalent value proxy, not booked revenue.


In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

QUEUE_PATH = Path("work/outputs/baseline_action_score.csv")
DATA_PATH = Path("data/raw/content_refresh_anonymized.csv")
ML08_PATH = Path("work/outputs/ml08_model_metrics.json")
ML09_PATH = Path("work/outputs/ml09_validation_metrics.json")
assert QUEUE_PATH.exists(), "Run the ML-07 baseline notebook first."
assert DATA_PATH.exists(), "Expected the anonymized source data."

queue = pd.read_csv(QUEUE_PATH)
raw = pd.read_csv(DATA_PATH, usecols=["content_id", "cpc", "main_intent", "competition_level", "trend_direction", "trend_pct"])
queue = queue.merge(raw.drop(columns=["trend_direction", "trend_pct"]), on="content_id", how="left", validate="one_to_one")

FORBIDDEN = {"trend_direction", "trend_pct", "observed_decline_outcome", "is_declining_label"}
assert not FORBIDDEN.intersection(queue.columns), "Retrospective or label-derived field entered the playbook export."
assert len(queue) == len(pd.read_csv(QUEUE_PATH))

queue["captured_value_proxy_90d"] = (queue["clicks_90d"].fillna(0) * queue["cpc"].fillna(0)).round(2)
queue["priority_tier"] = np.select(
    [queue["score"] >= 5, queue["score"] >= 3, queue["score"] >= 1],
    ["P1_review_this_week", "P2_review_next_cycle", "P3_monitor_or_sample"],
    default="P4_hold_without_new_evidence",
)

action_map = {
    "refresh_and_recheck": {
        "recommended_next_step": "Check freshness, search intent, facts, and cannibalization; refresh only if the review confirms an opportunity.",
        "estimated_effort_hours": 3.0,
        "human_review_gate": "editor + SEO reviewer",
    },
    "improve_snippet_or_intent": {
        "recommended_next_step": "Review title, description, opening answer, and query fit; propose a reversible snippet or relevance test.",
        "estimated_effort_hours": 1.5,
        "human_review_gate": "SEO reviewer",
    },
    "review_search_fit": {
        "recommended_next_step": "Inspect current search intent and competing coverage before proposing a targeted revision.",
        "estimated_effort_hours": 1.0,
        "human_review_gate": "SEO reviewer + editor",
    },
    "monitor": {
        "recommended_next_step": "Record the page in the monitoring sample; do not edit without a new signal.",
        "estimated_effort_hours": 0.25,
        "human_review_gate": "analyst",
    },
    "monitor_or_research": {
        "recommended_next_step": "Collect a small evidence sample and revisit after the next measurement window.",
        "estimated_effort_hours": 0.5,
        "human_review_gate": "analyst",
    },
}
for action, fields in action_map.items():
    mask = queue["action"].eq(action)
    for field, value in fields.items():
        queue.loc[mask, field] = value

queue["value_band"] = pd.cut(
    queue["captured_value_proxy_90d"],
    bins=[-np.inf, 1, 25, 250, np.inf],
    labels=["none_or_unknown", "low", "medium", "high"],
)
queue["no_go_without_human"] = True
queue = queue.sort_values(["priority_tier", "score", "captured_value_proxy_90d", "rank"], ascending=[True, False, False, True]).reset_index(drop=True)
queue["playbook_rank"] = np.arange(1, len(queue) + 1)

print(f"Queue rows: {len(queue):,}; base source rows: 30,000")
print("Priority tiers:")
print(queue["priority_tier"].value_counts().sort_index().to_string())
print("\nActions:")
print(queue["action"].value_counts().to_string())
print("\nTop 10 review queue:")
print(queue[["playbook_rank", "content_id", "action", "reason_code", "score", "priority_tier", "value_band", "captured_value_proxy_90d"]].head(10).to_string(index=False))


Queue rows: 30,000; base source rows: 30,000
Priority tiers:
priority_tier
P1_review_this_week              2182
P2_review_next_cycle            12008
P3_monitor_or_sample             6673
P4_hold_without_new_evidence     9137

Actions:
action
improve_snippet_or_intent    12597
monitor                      11248
monitor_or_research           5556
review_search_fit              577
refresh_and_recheck             22

Top 10 review queue:
 playbook_rank           content_id                    action             reason_code  score       priority_tier      value_band  captured_value_proxy_90d
             1 content_cf56e2e2e282       refresh_and_recheck   stale_visible_low_ctr      6 P1_review_this_week none_or_unknown                      0.00
             2 content_0a91db491d14       refresh_and_recheck   stale_visible_low_ctr      6 P1_review_this_week none_or_unknown                      0.00
             3 content_c2d929d83eaa       refresh_and_recheck   stale_visible_low_ctr      6 P

## 2. Intended use and limits

**Intended user:** an SEO/editorial analyst or content lead.

**Intended use:** select a small review queue from pages with observable prior visibility, low click capture, or staleness signals; record the reason code; check the page manually; and choose a reversible next step. The queue is most useful for deciding what deserves attention first, not for declaring a page good or bad.

**Limits:** the source is an anonymized snapshot; the score is retrospective and directional; `clicks × CPC` is a value proxy rather than revenue; the model evidence was evaluated on a grouped holdout but is not a production guarantee; and no intervention test shows that a refresh causes recovery. Seasonal demand, SERP changes, technical indexing, cannibalization, brand/legal constraints, and already-scheduled work can make a recommendation wrong.

The decay/refresh insight is also narrow: the research paper observed stronger outcomes for recently refreshed older pages, but refresh selection and survivor bias can explain part of that gap. Use it to prioritize inspection of mature, visible pages — not as a promise that refreshing every old page will improve performance.


In [2]:
# A compact playbook summary for reviewers and the next paper section.
summary = (
    queue.groupby(["priority_tier", "action"], observed=False)
    .agg(
        queue_rows=("content_id", "size"),
        median_score=("score", "median"),
        total_click_value_proxy=("captured_value_proxy_90d", "sum"),
        median_effort_hours=("estimated_effort_hours", "median"),
    )
    .reset_index()
)
print(summary.to_string(index=False, float_format=lambda v: f"{v:.2f}"))

# The queue remains decision-support: every row must carry a reason and review gate.
assert queue["reason_code"].notna().all()
assert queue["recommended_next_step"].notna().all()
assert queue["human_review_gate"].notna().all()
assert queue["no_go_without_human"].all()
print("\nIntended use check: every queued row has a reason code, next step, estimated effort, and human review gate.")


               priority_tier                    action  queue_rows  median_score  total_click_value_proxy  median_effort_hours
         P1_review_this_week improve_snippet_or_intent        2170          5.00                 35072.04                 1.50
         P1_review_this_week       refresh_and_recheck          12          5.00                     0.00                 3.00
        P2_review_next_cycle improve_snippet_or_intent       10427          4.00                 59651.95                 1.50
        P2_review_next_cycle       monitor_or_research        1202          3.00                 10766.20                 0.50
        P2_review_next_cycle       refresh_and_recheck          10          4.00                     0.00                 3.00
        P2_review_next_cycle         review_search_fit         369          3.00                 13643.32                 1.00
        P3_monitor_or_sample                   monitor        2111          1.00                   178.02      

## 3. Human review + the no-go list

Before anyone edits a page, the reviewer must check:

1. Whether the page is still live, indexed, and assigned to the right owner.
2. Whether the current query intent and competing results support the proposed action.
3. Whether the observed signal is seasonal, caused by a site-wide issue, or already addressed.
4. Whether facts, citations, claims, brand voice, accessibility, and legal/compliance constraints are safe.
5. Whether a related page is cannibalizing demand or a technical issue is the real blocker.
6. What success metric and 30/60-day follow-up window will be recorded.

### No-go without explicit human approval

- No automatic publishing, rewriting, deletion, redirect, canonical, `noindex`, or internal-link changes.
- No automatic claims that a refresh will increase traffic, rankings, or revenue.
- No automatic quality, author, safety, or policy verdict from a score.
- No use of retrospective outcome fields or product decision flags as hidden inputs.
- No prioritization that overrides editorial, legal, accessibility, or client instructions.
- No action on a low-score row merely to increase queue volume.


In [3]:
no_go_rules = [
    "publish_or_rewrite_without_human_approval",
    "delete_redirect_or_indexing_change_without_human_approval",
    "claim_causality_from_retrospective_association",
    "use_label_or_future_window_as_an_input",
    "override_editorial_legal_accessibility_or_client_constraints",
    "treat_score_as_content_quality_verdict",
]
review_checklist = [
    "page_live_and_owner_confirmed", "intent_and_competition_checked",
    "seasonality_and_sitewide_context_checked", "facts_citations_brand_and_legal_checked",
    "cannibalization_and_technical_blockers_checked", "success_metric_and_follow_up_window_recorded",
]
print("Human review checklist:")
for i, item in enumerate(review_checklist, 1): print(f"{i}. {item}")
print("\nNo-go rules:")
for item in no_go_rules: print(f"- {item}")
assert len(no_go_rules) >= 5 and len(review_checklist) >= 5


Human review checklist:
1. page_live_and_owner_confirmed
2. intent_and_competition_checked
3. seasonality_and_sitewide_context_checked
4. facts_citations_brand_and_legal_checked
5. cannibalization_and_technical_blockers_checked
6. success_metric_and_follow_up_window_recorded

No-go rules:
- publish_or_rewrite_without_human_approval
- delete_redirect_or_indexing_change_without_human_approval
- claim_causality_from_retrospective_association
- use_label_or_future_window_as_an_input
- override_editorial_legal_accessibility_or_client_constraints
- treat_score_as_content_quality_verdict


## 4. Monitoring / retrain triggers

This is a lightweight monitoring plan for a future workflow, not a claim that a production service exists.

- Recompute the base rate and Precision@10/@50/@100 on a fresh, time-ordered follow-up window every review cycle.
- Review a fixed sample of P1/P2 recommendations for reason-code accuracy and human accept/reject decisions.
- Trigger a model or rule review if grouped Precision@50 is below 0.55 for two consecutive cycles, or falls below the Week-4 baseline of 0.44.
- Trigger a data review if the decline base rate moves by more than 10 percentage points, a key feature's missingness exceeds 5%, or the content mix changes materially.
- Reconsider the action mapping if more than 30% of reviewed P1 rows are rejected for reasons not represented by the current codes.
- Retrain or revalidate only after a new labelled window exists; never retrain on the same retrospective window and call that a fresh evaluation.


In [4]:
ml08 = json.loads(ML08_PATH.read_text())
ml09 = json.loads(ML09_PATH.read_text())
monitoring = {
    "evaluation_window": "fresh time-ordered follow-up window; not available in this snapshot",
    "baseline_precision_at_50": 0.44,
    "grouped_model_precision_at_50_reference": ml08["comparison"][1]["precision_at_50"],
    "grouped_model_roc_auc_reference": ml08["comparison"][1]["roc_auc"],
    "trigger_precision_at_50_two_cycles_below": 0.55,
    "trigger_precision_at_50_below_week4_baseline": 0.44,
    "trigger_base_rate_shift_absolute": 0.10,
    "trigger_feature_missingness": 0.05,
    "trigger_rejected_p1_share": 0.30,
    "current_grouped_client_overlap": ml09["client_overlap"],
}
print(json.dumps(monitoring, indent=2))
assert monitoring["current_grouped_client_overlap"] == 0
print("\nMonitoring check: reference receipts loaded; proposed triggers are explicit and measurable.")


{
  "evaluation_window": "fresh time-ordered follow-up window; not available in this snapshot",
  "baseline_precision_at_50": 0.44,
  "grouped_model_precision_at_50_reference": 0.62,
  "grouped_model_roc_auc_reference": 0.534718517639614,
  "trigger_precision_at_50_two_cycles_below": 0.55,
  "trigger_precision_at_50_below_week4_baseline": 0.44,
  "trigger_base_rate_shift_absolute": 0.1,
  "trigger_feature_missingness": 0.05,
  "trigger_rejected_p1_share": 0.3,
  "current_grouped_client_overlap": 0
}

Monitoring check: reference receipts loaded; proposed triggers are explicit and measurable.


## 5. Exports for the paper

The notebook exports the full ranked queue as a CSV for the next paper section. The queue is intentionally ignored by git because it is a regenerated data artifact; the compact JSON receipt is committed so the paper can trace the counts and guardrails without committing bulk rows.


In [5]:
output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)
queue_export = output_dir / "ml10_action_queue.csv"
metrics_export = output_dir / "ml10_playbook_metrics.json"

export_columns = [
    "playbook_rank", "content_id", "action", "reason_code", "priority_tier",
    "recommended_next_step", "human_review_gate", "estimated_effort_hours",
    "value_band", "captured_value_proxy_90d", "no_go_without_human",
    "confidence_note", "what_would_make_it_wrong", "content_type", "main_intent",
    "competition_level", "impressions_90d", "clicks_90d", "ctr",
    "avg_position", "days_since_last_update", "content_age_days",
]
queue[export_columns].to_csv(queue_export, index=False)
metrics = {
    "queue_rows": int(len(queue)),
    "priority_counts": {str(k): int(v) for k, v in queue["priority_tier"].value_counts().sort_index().items()},
    "action_counts": {str(k): int(v) for k, v in queue["action"].value_counts().items()},
    "reason_counts": {str(k): int(v) for k, v in queue["reason_code"].value_counts().items()},
    "p1_rows": int((queue["priority_tier"] == "P1_review_this_week").sum()),
    "p1_click_value_proxy_total": float(queue.loc[queue["priority_tier"] == "P1_review_this_week", "captured_value_proxy_90d"].sum()),
    "queue_source": "work/outputs/baseline_action_score.csv; transparent Week-4 score",
    "model_evidence_reference": {
        "grouped_precision_at_50": float(ml08["comparison"][1]["precision_at_50"]),
        "week4_precision_at_50": float(ml08["comparison"][0]["precision_at_50"]),
        "grouped_client_overlap": int(ml09["client_overlap"]),
    },
    "forbidden_inputs_excluded": sorted(FORBIDDEN),
    "human_review_required_for_every_row": bool(queue["no_go_without_human"].all()),
    "export_note": "CSV is regenerated and intentionally ignored by git; this JSON is the committed receipt.",
}
metrics_export.write_text(json.dumps(metrics, indent=2))
print(f"Wrote {queue_export} ({queue_export.stat().st_size:,} bytes)")
print(f"Wrote {metrics_export}")


Wrote work/outputs/ml10_action_queue.csv (11,819,416 bytes)
Wrote work/outputs/ml10_playbook_metrics.json


## Self-check

- [x] Ranked queue has human-readable actions and reason codes.
- [x] Intended use, limits, decay/refresh caveat, cost/value proxy, and non-production scope are stated.
- [x] Human review gates and explicit no-go cases are listed.
- [x] Monitoring and retrain triggers are measurable and do not pretend a fresh evaluation exists.
- [x] The queue is exported to `work/outputs/ml10_action_queue.csv`; the compact JSON receipt is committed.
- [x] Retrospective outcome fields and label-derived fields are excluded from the export.
- [x] Notebook runs top to bottom without errors.
